# 🚀 MambaCenter: VisDrone Small Object Detection on Kaggle GPU
This notebook provides an automated, production-ready training pipeline for **MambaCenter** on Kaggle.

### 📌 Highlights & Kaggle Optimizations:
- **Architectures**: CenterNet Baseline (DLA-34 + DCNv2), CP-Mamba (B5'), IA-Mamba (A4'/A5'), Addon-VSS, Mamba-FPN.
- **Hardware Adaptation**: Automatically detects Tesla T4 (sm_75) / P100 / A100.
- **Memory Safety**: Uses batch size 8 with gradient accumulation (effective batch 16), micro-batching, and activation checkpointing so it runs smoothly within 16GB VRAM.
- **Precision**: Automatically selects native FP16 with GradScaler on Tesla T4 (or BF16 on A100/RTX).
- **Data & Pretrained Weights**: One-click VisDrone auto-linking, COCO conversion, and DLA-34 ImageNet caching.

## 1. Environment & GPU Inspection
Verifies CUDA, PyTorch, and GPU hardware capabilities.

In [ ]:
import torch
import os
import sys
import subprocess

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    print(f"Native bfloat16 hardware support: {bf16_ok}")
    print(f"Recommended Precision: {'bf16' if bf16_ok else 'fp16'}")
    !nvidia-smi
else:
    raise RuntimeError('No GPU detected! Go to right sidebar -> Settings -> Accelerator -> GPU T4 x2 or GPU P100.')


## 2. Obtain Codebase & Kaggle Hardware Configs
Links uploaded code dataset, extracts archive, or clones repo, then ensures all Kaggle configs, namespace fixes, and training optimizations are present.

In [ ]:
import os
import sys
import glob
import shutil
import zipfile
from pathlib import Path

WORKDIR = Path("/kaggle/working/MambaCenter")
WORKDIR.mkdir(parents=True, exist_ok=True)

# 1. Search for existing code in /kaggle/input (either unzipped dataset or zip file)
train_py_matches = [p for p in glob.glob("/kaggle/input/**/tools/train.py", recursive=True) if "working" not in p]
zip_matches = glob.glob("/kaggle/input/**/MambaCenter_Kaggle_Ready.zip", recursive=True)

if train_py_matches and not (WORKDIR / "models" / "mambacenter.py").exists() and not (WORKDIR / "mambacenter").exists():
    src_dir = Path(train_py_matches[0]).parent.parent
    print(f"Found unzipped code dataset at: {src_dir}")
    print(f"Copying codebase to {WORKDIR}...")
    shutil.copytree(src_dir, WORKDIR, dirs_exist_ok=True)
    print("Copied codebase successfully!")
elif zip_matches and not (WORKDIR / "tools" / "train.py").exists():
    print(f"Found zip archive at {zip_matches[0]}, extracting...")
    with zipfile.ZipFile(zip_matches[0], 'r') as zf:
        zf.extractall(WORKDIR)
    print("Extracted codebase successfully!")
elif not (WORKDIR / "tools" / "train.py").exists():
    print("Cloning repository from GitHub...")
    ret = os.system(f"git clone https://github.com/hurry211/MambaCenter-Research.git {WORKDIR}")
    if ret != 0 or not (WORKDIR / "tools" / "train.py").exists():
        !git clone https://github.com/lexiangrui/MambaCenter.git {WORKDIR}
else:
    print(f"Codebase already present at {WORKDIR}")

# 2. Guarantee that CenterNet submodules have __init__.py
for d in [WORKDIR / "baseline/CenterNet/src/lib/datasets", WORKDIR / "baseline/CenterNet/src/lib/models", WORKDIR / "baseline/CenterNet/src/lib/trains", WORKDIR / "baseline/CenterNet/src/lib/detectors", WORKDIR / "baseline/CenterNet/src/lib/utils"]:
    d.mkdir(parents=True, exist_ok=True)
    (d / "__init__.py").touch()
print("CenterNet package structure verified.")

# 3. Guarantee that Kaggle configs exist
configs_dir = WORKDIR / "configs"
configs_dir.mkdir(parents=True, exist_ok=True)

KAGGLE_T4_YAML = '''# Kaggle Tesla T4 (16GB) base training config - Plan B Every Epoch AP\nbase: _base.yaml\ntrain:\n  batch_size: 8\n  lr: 3.125e-5\n  amp: fp16\n  num_workers: 4\n  accum_steps: 2\neval:\n  interval: 1\n  batch_size: 8\n  workers: 4\n'''
KAGGLE_ADDON_CP_YAML = '''# CP-Mamba add-on (B5') for Kaggle Tesla T4 (16GB) - Plan B Every Epoch AP\nbase: addon_cp.yaml\ntrain:\n  batch_size: 8\n  lr: 6.25e-5\n  accum_steps: 2\n  amp: fp16\n  num_workers: 4\neval:\n  interval: 1\n  batch_size: 8\n  workers: 4\nmodel:\n  addon:\n    micro_batch: 4\n'''
KAGGLE_ADDON_IA_YAML = '''# IA-Mamba add-on (A4') for Kaggle Tesla T4 (16GB) - Plan B Every Epoch AP\nbase: addon_ia.yaml\ntrain:\n  batch_size: 8\n  lr: 6.25e-5\n  accum_steps: 2\n  amp: fp16\n  num_workers: 4\neval:\n  interval: 1\n  batch_size: 8\n  workers: 4\nmodel:\n  addon:\n    micro_batch: 4\n'''

for fn, content in [('kaggle_t4.yaml', KAGGLE_T4_YAML), ('kaggle_addon_cp.yaml', KAGGLE_ADDON_CP_YAML), ('kaggle_addon_ia.yaml', KAGGLE_ADDON_IA_YAML)]:
    cfg_p = configs_dir / fn
    cfg_p.write_text(content.strip() + '\n', encoding='utf-8')
    print(f"Config ready: {fn}")

# 4. Inject optimized tools/train.py, tools/common.py, tests/gpu_checks.py
(WORKDIR / "tools" / "train.py").write_text('#!/usr/bin/env python\n"""Train one CenterNet-family detector on VisDrone from an experiment config.\n\nEvery model uses CenterNet\'s VisDrone dataset and augmentation, CtdetLoss, decoding and\npost-processing. On top of CenterNet\'s training loop this adds bf16 autocast, a linear\nLR warm-up, an lr multiplier for parameters new to the DCN baseline (add-on models,\n``train.new_lr_mult``), the epoch hook of CP-Mamba\'s scan schedule, periodic COCO\nevaluation with explicit maxDets, a JSONL history and resumable checkpoints. A run\ndirectory is never reused: a new experiment needs a new name, continuing one needs --resume.\n\n    python tools/train.py --config configs/cpmamba.yaml [--set train.epochs=1 ...]\n"""\nimport argparse\nimport json\nimport os\nimport platform\nimport random\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport yaml\n\nimport common\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=__doc__,\n                                     formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument(\'--config\', help=\'experiment YAML (new runs)\')\n    parser.add_argument(\'--out\', help=\'run directory (default $MC_RUNS/<name>_s<seed>)\')\n    parser.add_argument(\'--resume\', action=\'store_true\',\n                        help=\'continue --out from its model_last.pth and config.yaml\')\n    parser.add_argument(\'--set\', nargs=\'*\', default=[], metavar=\'KEY=VALUE\',\n                        help=\'config overrides for a new run\')\n    parser.add_argument(\'--data-dir\', default=str(common.ROOT / \'data\'),\n                        help=\'directory containing visdrone/ (VisDrone2019-DET)\')\n    parser.add_argument(\'--max-iters\', type=int, default=0,\n                        help=\'iterations per epoch for smoke tests (0 = full epoch)\')\n    return parser.parse_args()\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n\n\ndef worker_init(_):\n    import cv2\n    cv2.setNumThreads(1)\n\n\ndef learning_rate(train, epoch, step):\n    """CenterNet step schedule (x0.1 after each lr_step epoch) with a linear warm-up."""\n    lr = float(train[\'lr\']) * 0.1 ** sum(epoch > s for s in train[\'lr_step\'])\n    warmup = int(train.get(\'warmup_iters\', 0))\n    return lr * (step + 1) / warmup if step < warmup else lr\n\n\ndef param_groups(model, train):\n    """Adam groups: everything at the schedule\'s lr, plus the parameters a model reports as\n    new to the baseline (``new_parameters``) at ``train.new_lr_mult`` times that lr."""\n    mult = float(train.get(\'new_lr_mult\', 1))\n    new = model.new_parameters() if mult != 1 and hasattr(model, \'new_parameters\') else []\n    ids = {id(p) for p in new}\n    groups = [{\'params\': [p for p in model.parameters() if id(p) not in ids], \'lr_mult\': 1.0}]\n    if new:\n        groups.append({\'params\': new, \'lr_mult\': mult})\n    return groups\n\n\ndef save(path, payload):\n    tmp = path.with_suffix(\'.tmp\')\n    torch.save(payload, tmp)\n    os.replace(tmp, path)\n\n\nclass Log:\n    def __init__(self, run):\n        self.history = open(run / \'history.jsonl\', \'a\')\n        self.text = open(run / \'log.txt\', \'a\')\n\n    def line(self, message):\n        print(message, flush=True)\n        self.text.write(message + \'\\n\')\n        self.text.flush()\n\n    def record(self, record):\n        self.history.write(json.dumps(record) + \'\\n\')\n        self.history.flush()\n\n\ndef train_epoch(model, loader, criterion, optimizer, cfg, epoch, step, log, max_iters, scaler=None):\n    train = cfg[\'train\']\n    device = torch.device(\'cuda\')\n    amp_opt = str(train.get(\'amp\', \'bf16\')).lower()\n    if amp_opt in (\'bf16\', \'true\') and not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()):\n        amp_opt = \'fp16\'\n    amp_enabled = amp_opt in (\'bf16\', \'fp16\')\n    amp_dtype = torch.bfloat16 if amp_opt == \'bf16\' else torch.float16\n    accum_steps = max(1, int(train.get(\'accum_steps\', 1)))\n    clip = train.get(\'clip_grad\')\n    model.train()\n    totals, count, skipped = defaultdict(float), 0, 0\n    torch.cuda.reset_peak_memory_stats()\n    start = data_end = time.time()\n    data_time = 0.0\n    optimizer.zero_grad(set_to_none=True)\n    for it, batch in enumerate(loader):\n        if max_iters and it >= max_iters:\n            break\n        lr = learning_rate(train, epoch, step)\n        for group in optimizer.param_groups:\n            group[\'lr\'] = lr * group.get(\'lr_mult\', 1.0)\n        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items() if k != \'meta\'}\n        data_time += time.time() - data_end\n        with torch.autocast(\'cuda\', dtype=amp_dtype, enabled=amp_enabled):\n            outputs = model(batch[\'input\'])\n        loss, stats = criterion(outputs, batch)\n        if not torch.isfinite(loss):\n            skipped += 1\n            log.line(\'epoch %d iter %d: non-finite loss, step skipped\' % (epoch, it))\n            if skipped > 20:\n                raise RuntimeError(\'more than 20 non-finite losses in one epoch\')\n            data_end = time.time()\n            continue\n        scaled_loss = loss / accum_steps if accum_steps > 1 else loss\n        if scaler is not None and scaler.is_enabled():\n            scaler.scale(scaled_loss).backward()\n        else:\n            scaled_loss.backward()\n        is_step = ((it + 1) % accum_steps == 0) or ((it + 1) == len(loader)) or (max_iters and (it + 1) >= max_iters)\n        if is_step:\n            if scaler is not None and scaler.is_enabled():\n                if clip:\n                    scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), float(clip))\n                scaler.step(optimizer)\n                scaler.update()\n            else:\n                if clip:\n                    torch.nn.utils.clip_grad_norm_(model.parameters(), float(clip))\n                optimizer.step()\n            optimizer.zero_grad(set_to_none=True)\n            step += 1\n        count += 1\n        for key, value in stats.items():\n            totals[key] += float(value.detach())\n        if it % train.get(\'log_interval\', 50) == 0:\n            elapsed = time.time() - start\n            log.line(\'epoch %d [%d/%d] lr %.3g | %s | %.1f img/s data %.0f%%\' % (\n                epoch, it, len(loader), lr,\n                \' \'.join(\'%s %.4f\' % (k, totals[k] / count) for k in sorted(totals)),\n                count * loader.batch_size / elapsed, 100 * data_time / elapsed))\n        data_end = time.time()\n    elapsed = time.time() - start\n    return {**{k: v / max(count, 1) for k, v in totals.items()}, \'iters\': count,\n            \'skipped\': skipped, \'lr\': lr, \'minutes\': elapsed / 60,\n            \'img_per_s\': count * loader.batch_size / elapsed,\n            \'max_mem_gib\': torch.cuda.max_memory_allocated() / 2 ** 30}, step\n\n\ndef main():\n    args = parse_args()\n    if args.resume:\n        if not args.out or args.config:\n            raise SystemExit(\'--resume takes --out and optional --set (the saved config is reused)\')\n        run = Path(args.out)\n        cfg = common.load_config(run / \'config.yaml\', args.set)\n        if args.set:\n            (run / \'config.yaml\').write_text(yaml.safe_dump(cfg, sort_keys=False))\n    else:\n        if not args.config:\n            raise SystemExit(\'--config is required for a new run\')\n        cfg = common.load_config(args.config, args.set)\n        run = Path(args.out) if args.out else common.run_dir(cfg)\n        if run.exists() and any(run.iterdir()):\n            raise SystemExit(\'%s exists: choose a new name or pass --resume\' % run)\n        run.mkdir(parents=True, exist_ok=True)\n        (run / \'config.yaml\').write_text(yaml.safe_dump(cfg, sort_keys=False))\n    common.setup_paths()\n    import evaluate\n    from trains.ctdet import CtdetLoss\n\n    log = Log(run)\n    seed_everything(cfg[\'seed\'])\n    opt, dataset_cls = common.centernet_opt(cfg, run, args.data_dir)\n    torch.backends.cudnn.benchmark = True\n    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = True\n    device = torch.device(\'cuda\')\n    model = common.create_detector(cfg, opt).to(device)\n    groups = param_groups(model, cfg[\'train\'])\n    optimizer = torch.optim.Adam(groups, lr=float(cfg[\'train\'][\'lr\']))\n    criterion = CtdetLoss(opt)\n    \n    amp_cfg = str(cfg[\'train\'].get(\'amp\', \'bf16\')).lower()\n    if amp_cfg in (\'bf16\', \'true\') and not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()):\n        gpu_name = torch.cuda.get_device_name() if torch.cuda.is_available() else \'unknown\'\n        log.line(\'[AMP Notice] GPU (%s) lacks native bf16 support; auto-fallback to fp16 with GradScaler.\' % gpu_name)\n        amp_cfg = \'fp16\'\n    scaler = torch.cuda.amp.GradScaler(enabled=(amp_cfg == \'fp16\'))\n\n    start_epoch, step = 0, 0\n    if args.resume:\n        checkpoint = torch.load(run / \'model_last.pth\', map_location=\'cpu\', weights_only=False)\n        model.load_state_dict(checkpoint[\'state_dict\'])\n        optimizer.load_state_dict(checkpoint[\'optimizer\'])\n        if \'scaler\' in checkpoint and checkpoint[\'scaler\'] is not None and scaler.is_enabled():\n            scaler.load_state_dict(checkpoint[\'scaler\'])\n        start_epoch, step = checkpoint[\'epoch\'], checkpoint[\'step\']\n        log.line(\'resumed %s at epoch %d\' % (run, start_epoch))\n    nodename = os.uname().nodename if hasattr(os, \'uname\') else platform.node()\n    env = {\'slurm_job\': os.environ.get(\'SLURM_JOB_ID\'), \'node\': nodename,\n           \'gpu\': torch.cuda.get_device_name(), \'torch\': torch.__version__,\n           \'cuda\': torch.version.cuda, \'params\': common.count_parameters(model),\n           \'lr_groups\': [{\'params\': sum(p.numel() for p in g[\'params\']), \'lr_mult\': g[\'lr_mult\']}\n                         for g in groups],\n           \'resumed_at_epoch\': start_epoch}\n    with open(run / \'env.jsonl\', \'a\') as stream:\n        stream.write(json.dumps(env) + \'\\n\')\n    log.line(\'run %s | arch %s | params %.2fM | lr groups %s | job %s on %s %s\' % (\n        cfg[\'name\'], opt.arch, env[\'params\'] / 1e6,\n        \', \'.join(\'%.2fM x%g\' % (g[\'params\'] / 1e6, g[\'lr_mult\']) for g in env[\'lr_groups\']),\n        env[\'slurm_job\'], env[\'node\'], env[\'gpu\']))\n\n    train = cfg[\'train\']\n    workers = int(train[\'num_workers\'])\n    loader = torch.utils.data.DataLoader(\n        dataset_cls(opt, \'train\'), batch_size=opt.batch_size, shuffle=True,\n        num_workers=workers, pin_memory=True, drop_last=True, worker_init_fn=worker_init,\n        persistent_workers=workers > 0, prefetch_factor=2 if workers > 0 else None)\n    ev = cfg[\'eval\']\n    evaluator = evaluate.Evaluator(opt, dataset_cls, \'val\', ev[\'batch_size\'], ev[\'workers\'],\n                                   ev[\'max_dets\'])\n    epochs = int(train[\'epochs\'])\n    best_val_score = -1.0\n    if args.resume and (run / \'history.jsonl\').exists():\n        try:\n            with open(run / \'history.jsonl\', \'r\', encoding=\'utf-8\') as f:\n                for line in f:\n                    rec = json.loads(line)\n                    v = rec.get(\'val\', {})\n                    s = v.get(\'AP50@500\', v.get(\'AP50\', -1.0))\n                    if s and s > best_val_score:\n                        best_val_score = s\n            log.line(\'Resumed best_val_score from history: %.4f\' % best_val_score)\n        except Exception as e:\n            log.line(\'Failed to parse history for best score: %s\' % e)\n    for epoch in range(start_epoch + 1, epochs + 1):\n        if hasattr(model, \'set_epoch\'):\n            model.set_epoch(epoch - 1)\n        stats, step = train_epoch(model, loader, criterion, optimizer, cfg, epoch, step, log,\n                                  args.max_iters, scaler=scaler)\n        record = {\'epoch\': epoch, \'step\': step, \'train\': stats}\n        save(run / \'model_last.pth\', {\'epoch\': epoch, \'step\': step, \'state_dict\': model.state_dict(),\n                                      \'optimizer\': optimizer.state_dict(),\n                                      \'scaler\': scaler.state_dict() if scaler.is_enabled() else None})\n        if epoch in train[\'lr_step\']:\n            save(run / (\'model_%d.pth\' % epoch), {\'epoch\': epoch, \'state_dict\': model.state_dict()})\n        final = epoch == epochs\n        if final or epoch % int(ev[\'interval\']) == 0:\n            metrics, summary = evaluator(model, device,\n                                         run / \'eval_val_dets.json\' if final else None)\n            record[\'val\'] = metrics\n            log.line(summary.strip())\n            val_score = metrics.get(\'AP50@500\', metrics.get(\'AP@500\', metrics.get(\'AP50@100\', -1.0)))\n            if val_score > best_val_score:\n                best_val_score = val_score\n                save(run / \'model_best.pth\', {\'epoch\': epoch, \'step\': step, \'state_dict\': model.state_dict(),\n                                              \'metrics\': metrics})\n                log.line(\'>>> [NEW BEST] AP50@500: %.4f -> saved model_best.pth\' % val_score)\n        log.record(record)\n        log.line(\'epoch %d done: %s\' % (epoch, json.dumps(\n            {k: float(\'%.4g\' % v) if isinstance(v, float) else v\n             for k, v in {**stats, **record.get(\'val\', {})}.items()})))\n        if final:\n            save(run / \'model_final.pth\', {\'epoch\': epochs, \'state_dict\': model.state_dict()})\n            result = {\'run\': cfg[\'name\'], \'arch\': opt.arch, \'epoch\': epochs, \'split\': \'val\',\n                      \'test_size\': \'fix_res %d\' % opt.input_res, \'decode_topk\': opt.K,\n                      \'params\': common.count_parameters(model), **metrics}\n            (run / \'eval_val.json\').write_text(json.dumps(result, indent=2) + \'\\n\')\n            log.line(\'final: \' + json.dumps(result))\n\n\nif __name__ == \'__main__\':\n    sys.exit(main())\n', encoding='utf-8')
(WORKDIR / "tools" / "common.py").write_text('"""Shared helpers for the MambaCenter tools: paths, YAML experiment configs, CenterNet opts."""\nimport copy\nimport os\nimport sys\nfrom pathlib import Path\n\nimport yaml\n\nROOT = Path(__file__).resolve().parents[1]\nCENTERNET_LIB = ROOT / \'baseline\' / \'CenterNet\' / \'src\' / \'lib\'\n\n\ndef run_dir(cfg):\n    """runs/<name>_s<seed> under $MC_RUNS (cluster: /opt) or the repository\'s runs/."""\n    root = Path(os.environ.get(\'MC_RUNS\', ROOT / \'runs\'))\n    return root / (\'%s_s%d\' % (cfg[\'name\'], cfg[\'seed\']))\n\n\ndef setup_paths():\n    """Make CenterNet\'s src/lib and the mambacenter package importable."""\n    for k in list(sys.modules.keys()):\n        if k == \'datasets\' or k.startswith(\'datasets.\'):\n            if not hasattr(sys.modules[k], \'dataset_factory\'):\n                del sys.modules[k]\n    for path in (str(CENTERNET_LIB), str(ROOT)):\n        if path in sys.path:\n            sys.path.remove(path)\n        sys.path.insert(0, path)\n\n\ndef deep_merge(base, override):\n    out = copy.deepcopy(base)\n    for key, value in override.items():\n        if isinstance(value, dict) and isinstance(out.get(key), dict):\n            out[key] = deep_merge(out[key], value)\n        else:\n            out[key] = copy.deepcopy(value)\n    return out\n\n\ndef load_config(path, overrides=()):\n    """Load an experiment YAML; ``base:`` inherits another file, ``a.b=v`` overrides keys."""\n    path = Path(path)\n    cfg = yaml.safe_load(path.read_text()) or {}\n    base = cfg.pop(\'base\', None)\n    if base:\n        cfg = deep_merge(load_config(path.parent / base), cfg)\n    for item in overrides:\n        key, sep, value = item.partition(\'=\')\n        if not sep:\n            raise ValueError(\'override must look like key.sub=value: %s\' % item)\n        node = cfg\n        *parents, leaf = key.split(\'.\')\n        for name in parents:\n            node = node.setdefault(name, {})\n        node[leaf] = yaml.safe_load(value)\n    return cfg\n\n\ndef centernet_opt(cfg, save_dir, data_dir):\n    """CenterNet opts for one experiment config; returns (opt, dataset class)."""\n    setup_paths()\n    from opts import opts\n    from datasets.dataset_factory import get_dataset\n\n    train, loss = cfg[\'train\'], cfg.get(\'loss\', {})\n    argv = [\'ctdet\', \'--dataset\', cfg.get(\'dataset\', \'visdrone\'), \'--arch\', cfg[\'arch\'],\n            \'--exp_id\', cfg[\'name\'], \'--input_res\', str(cfg[\'input_res\']),\n            \'--down_ratio\', \'4\', \'--head_conv\', str(cfg.get(\'head_conv\', 256)),\n            \'--K\', str(cfg[\'eval\'][\'K\']), \'--batch_size\', str(train[\'batch_size\']),\n            \'--lr\', repr(float(train[\'lr\'])), \'--lr_step\', \',\'.join(map(str, train[\'lr_step\'])),\n            \'--num_epochs\', str(train[\'epochs\']), \'--num_workers\', str(train[\'num_workers\']),\n            \'--seed\', str(cfg[\'seed\']), \'--gpus\', \'0\',\n            \'--hm_weight\', repr(float(loss.get(\'hm_weight\', 1))),\n            \'--wh_weight\', repr(float(loss.get(\'wh_weight\', 0.1))),\n            \'--off_weight\', repr(float(loss.get(\'off_weight\', 1)))]\n    argv += [str(a) for a in cfg.get(\'centernet_args\', [])]\n    opt = opts().parse(argv)\n    dataset = get_dataset(opt.dataset, opt.task)\n    opt = opts().update_dataset_info_and_set_heads(opt, dataset)\n    opt.data_dir = str(data_dir)\n    opt.save_dir = opt.debug_dir = str(save_dir)\n    opt.coarse_weight = float(loss.get(\'coarse_weight\', 0))\n    opt.coarse_hm = opt.coarse_weight > 0\n    opt.pred_weight = float(loss.get(\'pred_weight\', 0))\n    return opt, dataset\n\n\ndef create_detector(cfg, opt, pretrained=True):\n    """CenterNet create_model with the Mamba architectures registered from cfg[\'model\']."""\n    setup_paths()\n    from models.model import create_model\n    from mambacenter.centernet import register\n\n    model_cfg = copy.deepcopy(cfg.get(\'model\') or {})\n    model_cfg[\'pretrained\'] = pretrained\n    register(model_cfg)\n    return create_model(opt.arch, opt.heads, opt.head_conv)\n\n\ndef count_parameters(model):\n    return sum(p.numel() for p in model.parameters())\n', encoding='utf-8')
(WORKDIR / "tests" / "gpu_checks.py").write_text('#!/usr/bin/env python\n"""GPU checks before training (run through Slurm; writes a JSON report).\n\n1. native selective_scan_cuda vs the torch reference: outputs and every gradient,\n   grouped B/C, kernel softplus+bias and explicit Δ;\n2. CP-SSM and IA-SSM mixers: CUDA path vs reference path (outputs and gradients);\n3. CenterNet DCN on CUDA vs the float64 DCNv2 reference;\n4. CenterNet VisDrone samples: stride-8 coarse targets peak at every object centre;\n5. (--addon-configs) add-on detectors loaded with a trained DCN baseline checkpoint give\n   the baseline\'s outputs on real val images, in fp32 and under bf16 autocast.\n"""\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport torch\n\nROOT = Path(__file__).resolve().parents[1]\nCENTERNET_LIB = ROOT / \'baseline\' / \'CenterNet\' / \'src\' / \'lib\'\nfor k in list(sys.modules.keys()):\n    if k == \'datasets\' or k.startswith(\'datasets.\'):\n        if not hasattr(sys.modules[k], \'dataset_factory\'):\n            del sys.modules[k]\nsys.path[:0] = [str(CENTERNET_LIB), str(ROOT), str(ROOT / \'tools\'), str(ROOT / \'tests\')]\n\nfrom mambacenter import scan  # noqa: E402\nfrom mambacenter.cp_mamba import CPConfig, CPSS2D  # noqa: E402\nfrom mambacenter.ia_mamba import IAConfig, IASS2D  # noqa: E402\n\n\ndef max_rel(a, b):\n    return float((a.double() - b.double()).abs().max() / (b.double().abs().max() + 1e-12))\n\n\ndef check_kernel(report):\n    torch.manual_seed(0)\n    dev = torch.device(\'cuda\')\n    b, k, d, n, length = 2, 4, 16, 16, 777\n    for softplus in (True, False):\n        u = torch.randn(b, k * d, length, device=dev)\n        delta = torch.randn(b, k * d, length, device=dev) * 0.5\n        if not softplus:\n            delta = torch.nn.functional.softplus(delta)\n        A = -torch.rand(k * d, n, device=dev) * 2 - 0.05\n        B = torch.randn(b, k, n, length, device=dev)\n        C = torch.randn(b, k, n, length, device=dev)\n        D = torch.randn(k * d, device=dev)\n        bias = torch.randn(k * d, device=dev) * 0.1 if softplus else None\n        inputs = [t.requires_grad_() for t in (u, delta, A, B, C, D) + ((bias,) if softplus else ())]\n        grad = torch.randn(b, k * d, length, device=dev)\n        result = {}\n        for backend in (\'cuda\', \'torch\'):\n            y = scan.selective_scan(*inputs[:6], delta_bias=bias, delta_softplus=softplus,\n                                    backend=backend)\n            result[backend] = (y, torch.autograd.grad(y, inputs, grad))\n        errors = {\'out\': max_rel(result[\'cuda\'][0], result[\'torch\'][0])}\n        names = [\'u\', \'delta\', \'A\', \'B\', \'C\', \'D\', \'bias\']\n        for name, g_cuda, g_ref in zip(names, result[\'cuda\'][1], result[\'torch\'][1]):\n            errors[\'d\' + name] = max_rel(g_cuda, g_ref)\n        report[\'kernel_softplus\' if softplus else \'kernel_explicit_delta\'] = errors\n        assert max(errors.values()) < 2e-3, errors\n\n\ndef check_mixers(report):\n    dev = torch.device(\'cuda\')\n    torch.manual_seed(0)\n    cp = CPSS2D(32, CPConfig(extra_ratio=0.0), expand=1, d_state=16).to(dev).train()\n    cp.set_progress(100)\n    with torch.no_grad():\n        cp.alpha.fill_(0.7)\n    ia = IASS2D(32, 0, IAConfig(), expand=1, d_state=16).to(dev).train()\n    with torch.no_grad():\n        ia.gamma.fill_(0.5)\n        ia.P.normal_(std=0.05)\n    x = torch.randn(2, 24, 20, 32, device=dev)\n\n    def run(module, backend, aux):\n        module.backend = backend\n        xx = x.clone().requires_grad_()\n        y, *extras = module(xx, aux)\n        loss = y.square().mean() + sum(e.mean() for e in extras)  # IA: E and L_pred\n        params = [p for p in module.parameters() if p.requires_grad]\n        grads = torch.autograd.grad(loss, [xx] + params, allow_unused=True)\n        return y, grads\n\n    torch.manual_seed(1)\n    aux = {\'cp_ssm\': torch.randn(2, 10, 12, 10, device=dev),      # coarse heatmap logits\n           \'ia_ssm\': torch.rand(2, 1, 24, 20, device=dev) * 2}    # incoming innovation map\n    for name, module in ((\'cp_ssm\', cp), (\'ia_ssm\', ia)):\n        outs = {backend: run(module, backend, aux[name]) for backend in (\'cuda\', \'torch\')}\n        errs = [max_rel(outs[\'cuda\'][0], outs[\'torch\'][0])]\n        errs += [max_rel(a, b) for a, b in zip(outs[\'cuda\'][1], outs[\'torch\'][1])\n                 if a is not None and b is not None]\n        report[name] = {\'out\': errs[0], \'max_grad\': max(errs[1:])}\n        assert max(errs) < 5e-3, (name, errs)\n\n\ndef check_dcn(report):\n    from test_dcn import dcn_v2_reference\n    from torchvision.ops import deform_conv2d\n\n    torch.manual_seed(0)\n    cpu = [torch.randn(2, 8, 13, 11, dtype=torch.float64),\n           torch.randn(2, 18, 13, 11, dtype=torch.float64) * 2,\n           torch.rand(2, 9, 13, 11, dtype=torch.float64),\n           torch.randn(6, 8, 3, 3, dtype=torch.float64), torch.randn(6, dtype=torch.float64)]\n    cpu = [t.requires_grad_() for t in cpu]\n    gpu = [t.detach().float().cuda().requires_grad_() for t in cpu]\n    ref = dcn_v2_reference(*cpu)\n    out = deform_conv2d(gpu[0], gpu[1], gpu[3], gpu[4], padding=1, mask=gpu[2])\n    grad = torch.randn_like(ref)\n    g_ref = torch.autograd.grad(ref, cpu, grad)\n    g_out = torch.autograd.grad(out, gpu, grad.float().cuda())\n    errors = {\'out\': max_rel(out.cpu(), ref)}\n    errors.update({\'d%d\' % i: max_rel(a.cpu(), b) for i, (a, b) in enumerate(zip(g_out, g_ref))})\n    report[\'dcn_cuda_vs_dcnv2\'] = errors\n    assert max(errors.values()) < 1e-4, errors\n\n\ndef check_targets(report, config, data_dir):\n    import numpy as np\n    import common\n\n    cfg = common.load_config(config)\n    cfg[\'loss\'][\'coarse_weight\'] = 0.5\n    opt, dataset_cls = common.centernet_opt(cfg, \'/tmp\', data_dir)\n    dataset = dataset_cls(opt, \'train\')\n    rng = np.random.RandomState(0)\n    checked = 0\n    for index in rng.choice(len(dataset), 8, replace=False):\n        item = dataset[int(index)]\n        hm, hm_c = item[\'hm\'], item[\'hm_c\']\n        assert hm_c.shape == (hm.shape[0], hm.shape[1] // 2, hm.shape[2] // 2), hm_c.shape\n        w = hm.shape[2]\n        for ind in item[\'ind\'][item[\'reg_mask\'] == 1]:\n            y, x = divmod(int(ind), w)\n            assert hm[:, y, x].max() == 1 and hm_c[:, y // 2, x // 2].max() == 1\n            checked += 1\n    report[\'coarse_targets\'] = {\'objects_checked\': checked, \'max_objs\': dataset.max_objs}\n\n\ndef check_addon_parity(report, configs, baseline_ckpt, data_dir):\n    import common\n    import evaluate\n\n    def heads(model, images, amp):\n        with torch.no_grad(), torch.autocast(\'cuda\', dtype=torch.bfloat16, enabled=amp):\n            out = model(images)[-1]\n        return {k: out[k].float() for k in (\'hm\', \'wh\', \'reg\')}\n\n    state = torch.load(baseline_ckpt, map_location=\'cpu\', weights_only=False)[\'state_dict\']\n    base_cfg = common.load_config(ROOT / \'configs/centernet_dla34.yaml\')\n    opt, dataset_cls = common.centernet_opt(base_cfg, \'/tmp\', data_dir)\n    base = common.create_detector(base_cfg, opt)\n    base.load_state_dict(state)\n    base.cuda().eval()\n    loader = torch.utils.data.DataLoader(evaluate.EvalImages(dataset_cls(opt, \'val\'), opt),\n                                         batch_size=4, collate_fn=evaluate._collate)\n    images = next(iter(loader))[0].cuda()\n    ref = {amp: heads(base, images, amp) for amp in (False, True)}\n    for config in configs:\n        cfg = common.load_config(config)\n        copt, _ = common.centernet_opt(cfg, \'/tmp\', data_dir)\n        model = common.create_detector(cfg, copt, pretrained=False)\n        result = model.load_state_dict(state, strict=False)\n        new = {n for n, p in model.named_parameters()\n               if any(p is q for q in model.new_parameters())}\n        assert not result.unexpected_keys and set(result.missing_keys) <= new | {\n            n for n, _ in model.named_buffers()}, (result.unexpected_keys, result.missing_keys)\n        model.cuda().eval()\n        model.set_epoch(10 ** 6)          # CP-SSM priority route and Δ modulation fully on\n        errors = {}\n        for amp in (False, True):\n            out = heads(model, images, amp)\n            for key in out:\n                errors[\'%s_%s\' % (\'bf16\' if amp else \'fp32\', key)] = max_rel(out[key], ref[amp][key])\n        report[\'addon_parity_\' + cfg[\'name\']] = errors\n        assert max(errors.values()) < 1e-3, (config, errors)\n        del model\n        torch.cuda.empty_cache()\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--config\', default=str(ROOT / \'configs/centernet_dla34.yaml\'))\n    parser.add_argument(\'--data-dir\', default=str(ROOT / \'data\'))\n    parser.add_argument(\'--addon-configs\', nargs=\'*\', default=[])\n    parser.add_argument(\'--baseline-ckpt\', help=\'trained DCN baseline for the add-on parity check\')\n    parser.add_argument(\'--out\', required=True)\n    args = parser.parse_args()\n    report = {\'gpu\': torch.cuda.get_device_name(), \'torch\': torch.__version__,\n              \'scan_import_error\': scan.CUDA_IMPORT_ERROR}\n    for check in (check_kernel, check_mixers, check_dcn):\n        check(report)\n        print(check.__name__, \'ok\', flush=True)\n    check_targets(report, args.config, args.data_dir)\n    if args.addon_configs:\n        check_addon_parity(report, args.addon_configs, args.baseline_ckpt, args.data_dir)\n        print(\'check_addon_parity ok\', flush=True)\n    report[\'passed\'] = True\n    Path(args.out).write_text(json.dumps(report, indent=2) + \'\\n\')\n    print(json.dumps(report, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')
print("Injected updated train.py, common.py, and gpu_checks.py.")

# 5. Clean up any pre-existing HuggingFace datasets from sys.modules
for k in list(sys.modules.keys()):
    if k == 'datasets' or k.startswith('datasets.'):
        del sys.modules[k]

%cd {WORKDIR}


## 3. Install Python Dependencies
Removes colliding HuggingFace datasets library and installs CenterNet / Mamba requirements.

In [ ]:
# Remove colliding HuggingFace datasets to prevent namespace collision with CenterNet's datasets library
!pip uninstall -y -q datasets
!pip install -q ninja pycocotools einops timm ultralytics-thop easydict tensorboardX progress numba
print("Dependencies installed and verified.")


## 4. Compile Selective-Scan CUDA Extension
Compiles official Mamba `selective_scan_cuda` for Tesla T4 (sm_75) using `ninja`.

In [ ]:
import os
import sys

# Optimize compilation for Tesla T4 (Turing compute capability 7.5)
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MAX_JOBS"] = "2"

print("Compiling selective_scan_cuda (takes ~2 minutes)...\n")
!python runtime/selective-scan/build.py build_ext \
  --build-lib runtime/selective-scan/packages --build-temp runtime/selective-scan/build

# Set environment paths
os.environ["PYTHONPATH"] = f"{WORKDIR}/runtime/selective-scan/packages:{WORKDIR}/runtime/selective-scan/mamba:{WORKDIR}"
for p in [str(WORKDIR), str(WORKDIR / "runtime/selective-scan/packages"), str(WORKDIR / "runtime/selective-scan/mamba")]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Verify selective scan kernel
try:
    import selective_scan_cuda
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    print("\n[OK] selective_scan_cuda extension compiled and imported successfully!")
except Exception as e:
    print(f"\n[Warning] Could not import selective_scan_cuda: {e}")


## 5. Download Pretrained DLA-34 Backbone Weights
Caches ImageNet pretrained weights into `$TORCH_HOME/hub/checkpoints`.

In [ ]:
import os
import glob
import shutil
import urllib.request
from pathlib import Path

hub_dir = Path("/root/.cache/torch/hub/checkpoints")
hub_dir.mkdir(parents=True, exist_ok=True)
ckpt_file = hub_dir / "dla34-ba72cf86.pth"
local_prepackaged = WORKDIR / "weights" / "dla34-ba72cf86.pth"
input_weights = glob.glob("/kaggle/input/**/dla34-ba72cf86.pth", recursive=True)

if not ckpt_file.exists() or ckpt_file.stat().st_size < 1000000:
    if local_prepackaged.exists() and local_prepackaged.stat().st_size > 1000000:
        print(f"Using pre-packaged DLA-34 weights from {local_prepackaged}...")
        shutil.copyfile(local_prepackaged, ckpt_file)
    elif input_weights and Path(input_weights[0]).stat().st_size > 1000000:
        print(f"Using DLA-34 weights from /kaggle/input: {input_weights[0]}...")
        shutil.copyfile(input_weights[0], ckpt_file)
    else:
        url = "https://huggingface.co/timm/dla34.in1k/resolve/main/pytorch_model.bin"
        print(f"Downloading DLA-34 checkpoint from {url} to {ckpt_file}...")
        urllib.request.urlretrieve(url, ckpt_file)
    print(f"DLA-34 ImageNet checkpoint cached successfully ({ckpt_file.stat().st_size / (1024**2):.2f} MB)!")
else:
    print("DLA-34 checkpoint already exists.")


## 6. VisDrone Dataset Setup & COCO Conversion
Locates VisDrone in `/kaggle/input`, symlinks images to avoid duplicating disk space, and converts annotations via `tools/visdrone2coco.py`.

In [ ]:
import os
import glob
from pathlib import Path

visdrone_dir = WORKDIR / "data" / "visdrone"
visdrone_dir.mkdir(parents=True, exist_ok=True)

# Locate VisDrone2019-DET train & val in /kaggle/input
train_candidates = [p for p in glob.glob('/kaggle/input/**/VisDrone2019-DET-train', recursive=True) if os.path.isdir(p)]
val_candidates = [p for p in glob.glob('/kaggle/input/**/VisDrone2019-DET-val', recursive=True) if os.path.isdir(p)]

if train_candidates and val_candidates:
    train_src = Path(train_candidates[0])
    val_src = Path(val_candidates[0])
    print(f"Found train directory: {train_src}")
    print(f"Found val directory:   {val_src}")
    train_dst = visdrone_dir / "VisDrone2019-DET-train"
    val_dst = visdrone_dir / "VisDrone2019-DET-val"
    if not train_dst.exists():
        os.symlink(train_src, train_dst)
    if not val_dst.exists():
        os.symlink(val_src, val_dst)
        
    train_json = visdrone_dir / "visdrone2019_det_train_coco.json"
    val_json = visdrone_dir / "visdrone2019_det_val_coco.json"
    if not (train_json.exists() and val_json.exists() and train_json.stat().st_size > 1000):
        print("Converting VisDrone annotations to COCO json (takes ~30s)...")
        !python tools/visdrone2coco.py --root {visdrone_dir} --splits train val
    print("Dataset ready in data/visdrone!")
else:
    raise RuntimeError("VisDrone dataset not found in /kaggle/input! Please ensure 'abhimanyubhowmik1/visdrone' is added.")


## 7. Pre-Flight Verification Tests
Verifies CPU unit tests and GPU numerical equivalence.

In [ ]:
!python -m unittest discover -s tests -p 'test_*.py'
!python tests/gpu_checks.py --out checks.json
print("All verification checks passed!")


## 8. Quick Smoke Test (10 Iterations)
Confirms GPU VRAM usage and forward/backward/loss sanity before launching full training.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

SMOKE_DIR = Path("/kaggle/working/runs/smoke_test")
if SMOKE_DIR.exists():
    shutil.rmtree(SMOKE_DIR, ignore_errors=True)

smoke_cmd = f"{sys.executable} tools/train.py --config configs/kaggle_addon_cp.yaml --out {SMOKE_DIR} --max-iters 10 --set train.epochs=1 eval.interval=1000"
print(f"Running smoke test: {smoke_cmd}")
ret = os.system(smoke_cmd)
if ret != 0:
    raise RuntimeError(f"Smoke test failed with return code {ret}")
print("Smoke test passed successfully!")


## 9. Main Training Loop
Select your target model configuration and start training.
- `configs/kaggle_addon_cp.yaml` : CP-Mamba Add-on (B5')
- `configs/kaggle_addon_ia.yaml` : IA-Mamba Add-on (A4')
- `configs/kaggle_t4.yaml`       : CenterNet DLA-34 Baseline

In [ ]:
import os
import sys
import glob
import shutil
from pathlib import Path

CONFIG = "configs/kaggle_addon_cp.yaml"
RUN_NAME = "cpmamba_kaggle_t4_s317"
OUT_DIR = f"/kaggle/working/runs/{RUN_NAME}"
resume_ckpt = f"{OUT_DIR}/model_last.pth"

# Restore existing checkpoint from /kaggle/input if available
ckpt_matches = [p for p in glob.glob("/kaggle/input/**/model_last.pth", recursive=True) if "working" not in p]
if ckpt_matches and not os.path.exists(resume_ckpt):
    src_dir = Path(ckpt_matches[0]).parent
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"Restoring checkpoint files from {src_dir} to {OUT_DIR}...")
    for item in os.listdir(src_dir):
        s_item = src_dir / item
        d_item = Path(OUT_DIR) / item
        if s_item.is_file():
            shutil.copy2(s_item, d_item)
            print(f"  Restored {item} ({d_item.stat().st_size / 1024 / 1024:.2f} MB)")

if os.path.exists(resume_ckpt):
    print(f"Found existing checkpoint. Resuming {RUN_NAME} (targeting 15 epochs, Plan B: Every Epoch AP)...")
    train_cmd = f"{sys.executable} tools/train.py --resume --out {OUT_DIR} --set train.epochs=15 eval.interval=1"
else:
    if os.path.exists(OUT_DIR) and not os.path.exists(f"{OUT_DIR}/model_final.pth"):
        shutil.rmtree(OUT_DIR, ignore_errors=True)
    print(f"Starting fresh training for {RUN_NAME} (15 epochs, Plan B: Every Epoch AP)...")
    train_cmd = f"{sys.executable} tools/train.py --config {CONFIG} --out {OUT_DIR} --set train.epochs=15 eval.interval=1"

print(f"Executing: {train_cmd}")
ret = os.system(train_cmd)
if ret != 0:
    raise RuntimeError(f"Training loop exited with code {ret}")


## 10. Evaluation & Visual Analysis
Evaluates COCO AP on the validation set, plots training curves, and renders predictions.

In [ ]:
import json
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt

run_path = Path(OUT_DIR)
if (run_path / 'config.yaml').exists():
    ckpt_name = 'model_best.pth' if (run_path / 'model_best.pth').exists() else ('model_final.pth' if (run_path / 'model_final.pth').exists() else 'model_last.pth')
    if (run_path / ckpt_name).exists():
        print(f"Evaluating best checkpoint {ckpt_name} on VisDrone validation set...")
        eval_cmd = f"{sys.executable} tools/evaluate.py --run {OUT_DIR} --ckpt {ckpt_name}"
        os.system(eval_cmd)
    
    history_file = run_path / 'history.jsonl'
    if history_file.exists():
        records = [json.loads(line) for line in history_file.read_text().splitlines() if line.strip()]
        epochs = [r['epoch'] for r in records]
        losses = [r['train'].get('loss', 0) for r in records]
        
        print("\n" + "=" * 85)
        print("🎯 TRAINING PROGRESS & VALIDATION METRICS (PLAN B):")
        print("=" * 85)
        print(f"{'Epoch':<6} | {'Loss':<10} | {'AP@100':<10} | {'AP50@100':<10} | {'AP@500':<10} | {'AP50@500':<10} | {'img/s':<8}")
        print("-" * 85)
        for r in records:
            ep = r['epoch']
            l = r['train'].get('loss', 0.0)
            spd = r['train'].get('img_per_s', 0.0)
            val = r.get('val', {})
            def fmt(val, k):
                v = val.get(k, '-')
                return f"{v:.4f}" if isinstance(v, float) else str(v)
            ap100 = fmt(val, 'AP@100')
            ap50_100 = fmt(val, 'AP50@100')
            ap500 = fmt(val, 'AP@500')
            ap50_500 = fmt(val, 'AP50@500')
            print(f"{ep:<6} | {l:<10.4f} | {ap100:<10} | {ap50_100:<10} | {ap500:<10} | {ap50_500:<10} | {spd:<8.1f}")
        print("=" * 85 + "\n")
        
        plt.figure(figsize=(9, 4))
        plt.plot(epochs, losses, marker='o', color='#1f77b4', label='Training Loss')
        plt.title(f'Training Loss - {RUN_NAME}')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend()
        plt.savefig(run_path / 'loss_curve.png', bbox_inches='tight')
        plt.show()
        
    if (run_path / ckpt_name).exists():
        print("Rendering detections on validation images...")
        vis_cmd = f"{sys.executable} tools/visualize.py --run {OUT_DIR} --ckpt {ckpt_name} --n 4 --thresh 0.3"
        os.system(vis_cmd)
else:
    print(f'Run directory {OUT_DIR} not found. Skipping evaluation.')
